#### Load Dataset

In [2]:
import pandas as pd
import ast

data_path = "../data/usc-x-24-us-election/part_1/"

files = [
    "may_july_chunk_1.csv.gz",
    "may_july_chunk_2.csv.gz",
    "may_july_chunk_3.csv.gz",
    "may_july_chunk_4.csv.gz",
    "may_july_chunk_5.csv.gz"
]

dfs = []
for f in files:
    temp = pd.read_csv(data_path + f, compression='gzip')
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
df.shape

(250000, 32)

In [3]:
df_sample = df[['id', 'text']].dropna().sample(30, random_state=42)
df_sample = df_sample.reset_index(drop=True)
df_sample.shape

(30, 2)

In [4]:
gallup_issues_map = {
    "The economy": (
        "inflation is killing us, prices are too high, cost of living, grocery prices, gas prices, "
        "rent is too expensive, unemployment, jobs report, wages, economy is tanking, recession, "
        "stock market, interest rates, fed reserve, bidenomics, economic growth, GDP, shrinkflation, "
        "can't afford groceries, everything costs more, 5x what it was, printing money"
    ),
    "Democracy in the U.S.": (
        "election integrity, stolen election, voter fraud, rigged election, January 6th, "
        "threat to democracy, MAGA, Trump 2024, Biden 2024, presidential election, voting rights, "
        "ballot harvesting, mail-in ballots, deep state, political corruption, drain the swamp, "
        "Democrat agenda, Republican agenda, two-party system, political polarization, "
        "Trump should be president, Biden should resign, vote them out"
    ),
    "Terrorism and national security": (
        "domestic terrorist attack, ISIS, Al Qaeda, jihad, sleeper cells, "
        "terror on American soil, FBI terror watch list, radical extremists, "
        "homeland security threat — NOT Russia, NOT Ukraine, NOT China, NOT Israel"
    ),
    "Types of Supreme Court justices candidates would pick": (
        "Supreme Court nominee, SCOTUS pick, judicial appointment, originalist judge, "
        "liberal judge, conservative justice, lifetime appointment, Roe v Wade overturned, "
        "packing the court, court packing, judicial philosophy, constitutional interpretation, "
        "who will they nominate, next Supreme Court justice"
    ),
    "Immigration": (
        "open borders, illegal aliens, undocumented immigrants, border wall, border security, "
        "migrants crossing the border, asylum seekers, deportation, ICE, DHS, sanctuary cities, "
        "Biden opened the border, invasion at the border, Title 42, mass migration, "
        "illegal immigration crisis, border patrol, cartel, gotaways, fentanyl at the border"
    ),
    "Education": (
        "public schools, student loan forgiveness, college tuition, school debt, "
        "critical race theory in schools, parents rights in education, school board, "
        "teachers union, homeschooling, college affordability, school choice, vouchers, "
        "indoctrinating kids, curriculum, common core, DEI on campus"
    ),
    "Healthcare": (
        "healthcare costs, medical bills, health insurance, Obamacare, ACA, "
        "Medicare for all, Medicaid, prescription drug prices, insulin costs, "
        "can't afford healthcare, hospital bills, pre-existing conditions, "
        "universal healthcare, Big Pharma, drug companies, mental health crisis"
    ),
    "Gun policy": (
        "Second Amendment, gun rights, gun control, mass shooting, AR-15, "
        "assault weapons ban, background checks, red flag laws, NRA, "
        "right to bear arms, gun violence, school shooting, taking our guns, "
        "armed citizens, concealed carry, gun free zones don't work"
    ),
    "Abortion": (
        "abortion rights, pro-life, pro-choice, Roe v Wade, abortion ban, "
        "right to choose, killing babies, women's reproductive rights, "
        "abortion access, heartbeat bill, late term abortion, abortion clinic, "
        "planned parenthood, defund planned parenthood, life begins at conception"
    ),
    "Taxes": (
        "tax hikes, raising taxes, tax cuts, corporate tax, billionaire tax, "
        "IRS, paying too much in taxes, tax the rich, Trump tax cuts, "
        "death tax, estate tax, government taking my money, tax and spend, "
        "fair share in taxes, fiscal policy, tax burden"
    ),
    "Crime": (
        "crime is out of control, defund the police, back the blue, law and order, "
        "violent crime, murder rate, carjacking, shoplifting, soft on crime, "
        "criminal justice reform, prison reform, police brutality, "
        "George Soros DA, revolving door justice, catch and release, "
        "smash and grab, crime wave, fentanyl deaths"
    ),
    "Distribution of income and wealth in the U.S.": (
        "wealth inequality, the rich keep getting richer, income gap, "
        "billionaires vs working class, Wall Street greed, corporate greed, "
        "wealth gap, working class struggles, middle class is dying, "
        "1 percent owns everything, economic inequality, living paycheck to paycheck, "
        "CEOs making millions while workers suffer"
    ),
    "The federal budget deficit": (
        "national debt, deficit spending, government spending out of control, "
        "trillions in debt, wasteful spending, government waste, bloated budget, "
        "debt ceiling, fiscal responsibility, spending taxpayer money, "
        "printing money, borrowed from China, out of control spending, pork barrel"
    ),
    "Foreign affairs": (
        "U.S. foreign policy, America's role in the world, foreign aid, "
        "giving money to other countries, America first, globalism, "
        "United Nations, NATO allies, military intervention, regime change, "
        "why are we policing the world, sending billions overseas, diplomacy"
    ),
    "Situation in Middle East between Israelis and Palestinians": (
        "Israel Gaza war, Hamas October 7th attack, Palestinian civilians, "
        "IDF, Netanyahu, bombing Gaza, hostages in Gaza, ceasefire, "
        "free Palestine, pro-Israel, humanitarian crisis in Gaza "
        "— NOT Russia, NOT China, NOT Ukraine"
    ),
    "Energy policy": (
        "gas prices, oil prices, drill baby drill, keystone pipeline, "
        "green new deal, solar panels, wind energy, fossil fuels, "
        "energy independence, OPEC, electric vehicles, EV mandate, "
        "energy crisis, natural gas, fracking, carbon tax, energy costs"
    ),
    "Relations with Russia": (
        "Putin, Russian invasion of Ukraine, Zelensky, NATO vs Russia, "
        "sending weapons to Ukraine, funding Ukraine, Russian military aggression, "
        "nuclear threat from Russia, Kremlin, U.S. sanctions on Russia, "
        "war in Eastern Europe — NOT China, NOT Middle East, NOT Israel"
    ),
    "Race relations": (
        "systemic racism, white privilege, Black Lives Matter, BLM, "
        "racial inequality, racial justice, civil rights, police racism, "
        "critical race theory, reverse racism, affirmative action, "
        "racial divide, white supremacy, DEI, diversity equity inclusion, "
        "reparations, discrimination, racial profiling"
    ),
    "Relations with China": (
        "CCP, Chinese Communist Party, Taiwan invasion, TikTok ban, "
        "spy balloon, Chinese espionage, trade war with China, fentanyl from China, "
        "South China Sea, standing up to China, Xi Jinping "
        "— NOT Russia, NOT Ukraine, NOT Middle East"
    ),
    "Trade with other nations": (
        "tariffs, trade war, trade deficit, American jobs going overseas, "
        "outsourcing, bring jobs back to America, free trade, NAFTA, USMCA, "
        "imports and exports, trade deal, protecting American industry, "
        "buy American, manufacturing jobs, offshoring, trade imbalance"
    ),
    "Climate change": (
        "climate change is a hoax, global warming, carbon emissions, "
        "green new deal, renewable energy, climate crisis, net zero, "
        "Paris agreement, carbon footprint, electric vehicles, "
        "extreme weather, wildfires, sea level rising, climate activists, "
        "fossil fuels vs clean energy, ESG"
    ),
    "Transgender rights": (
        "transgender athletes, trans women in women's sports, gender identity, "
        "gender affirming care, puberty blockers, drag shows, "
        "protecting children from gender ideology, bathroom bills, "
        "nonbinary, pronouns, LGBTQ rights, trans rights are human rights, "
        "don't say gay, parental rights vs gender ideology"
    ),
}
gallup_issues = list(gallup_issues_map.keys())
gallup_descriptions = list(gallup_issues_map.values())

issues_text = "\n".join(gallup_issues)
print(issues_text)

The economy
Democracy in the U.S.
Terrorism and national security
Types of Supreme Court justices candidates would pick
Immigration
Education
Healthcare
Gun policy
Abortion
Taxes
Crime
Distribution of income and wealth in the U.S.
The federal budget deficit
Foreign affairs
Situation in Middle East between Israelis and Palestinians
Energy policy
Relations with Russia
Race relations
Relations with China
Trade with other nations
Climate change
Transgender rights


#### Create Issue Embeddings

In [5]:
import ollama
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# call Ollama once for all tweets
def embed_batch(texts):
    response = ollama.embed(
        model="nomic-embed-text",
        input=texts
    )
    return np.array(response["embeddings"])

gallup_embeddings = embed_batch(gallup_descriptions)

In [6]:
emb_path = "../data/gallup_embeddings.npy"

try:
    gallup_embeddings = np.load(emb_path)
    print("Loaded Gallup embeddings from file.")
except FileNotFoundError:
    print("Embeddings not found. Computing embeddings...")
    gallup_embeddings = embed_batch(gallup_descriptions)
    
    try:
        np.save(emb_path, gallup_embeddings)
        print("Saved embeddings to:", emb_path)
    except Exception as e:
        print("Error while saving embeddings:", e)

Loaded Gallup embeddings from file.


#### Clean Tweet

In [7]:
import re

def clean_tweet(text):
    if text is None:
        return ""
    
    text = re.sub(r'http\S+', '', text)     # remove URLs
    text = re.sub(r'@\w+', '', text)        # remove mentions
    text = re.sub(r'#', '', text)           # remove hashtag symbol
    text = re.sub(r'\n', ' ', text)         # remove newlines
    text = re.sub(r'\s+', ' ', text)        # remove extra spaces
    #text = re.sub(r'[^A-Za-z0-9 ]+', ' ', text) # remove punctuation, symbols, emojis
    text = text.lower().strip()
    
    return text

#### Classify Tweets by Similarity

In [8]:
def classify_tweet(tweet: str):
    
    tweet_embedding = embed_batch(tweet)
    similarities: np.ndarray = cosine_similarity(tweet_embedding, gallup_embeddings)[0]
    best_idx = similarities.argmax()

    best_score = similarities[best_idx]
    
    return gallup_issues[best_idx], best_score

In [9]:
### testing on 1 row of df
print(df_sample['text'].iloc[0])
print(classify_tweet(df_sample['text'].iloc[0]))

@kangaroos991 Everything else is in the past. So be it, what you choose to think. But how has Biden saved anything? The most common items are now 5x what they were 4 years ago. He’s just giving trillions away to other nations. Didn’t help Maui at all. Killed Americans, and opened the border???
('Immigration', np.float64(0.6503112115489431))


#### Run on 30 tweets pilot dataframe

In [10]:
df_sample['clean_text'] = df_sample['text'].apply(clean_tweet)

results = df_sample['clean_text'].apply(classify_tweet)
df_sample[['issue_semantic', 'similarity']] = pd.DataFrame(results.tolist(), index=df_sample.index)
avg_similarity = df_sample['similarity'].mean()
print("Average similarity:", df_sample['similarity'].mean())
print("Max similarity:", df_sample['similarity'].max())
print("Min similarity:", df_sample['similarity'].min())

Average similarity: 0.5543208565638377
Max similarity: 0.7080107272882599
Min similarity: 0.46851158898428535


In [11]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
df_sample.head(5)

,id,text,clean_text,issue_semantic,similarity
0,1801009467170943373,"@kangaroos991 Everything else is in the past. So be it, what you choose to think. But how has Biden saved anything? The most common items are now 5x what they were 4 years ago. He’s just giving trillions away to other nations. Didn’t help Maui at all. Killed Americans, and opened the border???","everything else is in the past. so be it, what you choose to think. but how has biden saved anything? the most common items are now 5x what they were 4 years ago. he’s just giving trillions away to other nations. didn’t help maui at all. killed americans, and opened the border???",Foreign affairs,0.514192
1,1800990362770509922,"Man who manipulates rates, thus inflation to help Biden get re-elected wonders why so many are mad?","man who manipulates rates, thus inflation to help biden get re-elected wonders why so many are mad?",The economy,0.627143
2,1801038374267969811,@Coloradosearch1 @JoJoFromJerz But unlike Donald Trump he is not a CONVICTED FELON or rapist,but unlike donald trump he is not a convicted felon or rapist,Crime,0.496728
3,1800948689441259936,@TrumpSolMAGA @WhaleInsider preparing for a massive bull 🔥$maga,preparing for a massive bull 🔥$maga,Relations with China,0.468512
4,1800910374780481866,"Biden reportedly puts the blames for Hunter's conviction on his re-election bid. Saying, ""He would have gotten the plea deal.""\nNow, that claim didn't work for Trump, when Trump's case was really happening.\nIf Biden thinks this will fly W/ voters, he's got another thing coming.","biden reportedly puts the blames for hunter's conviction on his re-election bid. saying, ""he would have gotten the plea deal."" now, that claim didn't work for trump, when trump's case was really happening. if biden thinks this will fly w/ voters, he's got another thing coming.",Crime,0.518298


In [12]:
try:
    df_sample.to_csv("../data/sample_issue_classification.csv", index=False)
    print("Saved dataset: ../data/sample_issue_classification.csv")
except Exception as e:
    print("Error while saving:", e)

Saved dataset: ../data/sample_issue_classification.csv


#### Run on entire df

In [13]:
# Clean tweets
df['clean_text'] = df['text'].apply(clean_tweet)

# Embed all tweets
tweet_embeddings = embed_batch(df['clean_text'].tolist())

# Similarity matrix
similarities = cosine_similarity(tweet_embeddings, gallup_embeddings)

# Best issue per tweet
best_indices = similarities.argmax(axis=1)
best_scores = similarities.max(axis=1)

# Assign back to dataframe
df['issue_semantic'] = [gallup_issues[i] for i in best_indices]
df['similarity'] = best_scores

# Stats
print("Average similarity:", df['similarity'].mean())
print("Max similarity:", df['similarity'].max())
print("Min similarity:", df['similarity'].min())

Average similarity: 0.5476769367762023
Max similarity: 0.8492825928245274
Min similarity: 0.32302618459846894


In [14]:
print(df.columns)

Index(['Unnamed: 0', 'id', 'text', 'url', 'epoch', 'media', 'retweetedTweet',
       'retweetedTweetID', 'retweetedUserID', 'id_str', 'lang', 'rawContent',
       'replyCount', 'retweetCount', 'likeCount', 'quoteCount',
       'conversationId', 'conversationIdStr', 'hashtags', 'mentionedUsers',
       'links', 'viewCount', 'quotedTweet', 'in_reply_to_screen_name',
       'in_reply_to_status_id_str', 'in_reply_to_user_id_str', 'location',
       'cash_app_handle', 'user', 'date', '_type', 'type', 'clean_text',
       'issue_semantic', 'similarity'],
      dtype='str')


In [16]:
print(type(df['hashtags'].iloc[0]))
print(repr(df['hashtags'].iloc[0]))

<class 'str'>
'[]'


In [17]:
cols_to_save = [
    'id',
    'user',
    'hashtags',
    'clean_text',
    'issue_semantic',
    'similarity'
]

try:
    df[cols_to_save].to_csv("../data/tweet_issue_labels.csv", index=False)
    print("Saved dataset: ../data/tweet_issue_labels.csv")
except Exception as e:
    print("Error while saving:", e)

# for merging back with original df later
# labels = pd.read_csv("../data/tweet_issue_labels.csv")
# df_full = df_full.merge(labels, on='id', how='left')

Saved dataset: ../data/tweet_issue_labels.csv


In [18]:
issue_summary = pd.DataFrame({
    'Tweets': df.groupby('issue_semantic').size(),
    'Users': df.groupby('issue_semantic')['user'].nunique(),
    'Avg Similarity': df.groupby('issue_semantic')['similarity'].mean()
})

issue_summary = issue_summary.sort_values(by='Tweets', ascending=False)

# Reset index for issue-column 
issue_summary = issue_summary.reset_index()
issue_summary = issue_summary.rename(columns={'issue_semantic': 'Issue'})

# Metrics
issue_summary['Avg Similarity'] = issue_summary['Avg Similarity'].round(3)
issue_summary['Tweets per User'] = (issue_summary['Tweets'] / issue_summary['Users']).round(2)
issue_summary['Tweet %'] = (issue_summary['Tweets'] / issue_summary['Tweets'].sum() * 100).round(2)
issue_summary['User %'] = (issue_summary['Users'] / issue_summary['Users'].sum() * 100).round(2)

# Reorder columns (for readability)
issue_summary = issue_summary[
    ['Issue', 'Tweets', 'Users', 'Tweets per User', 'Avg Similarity', 'Tweet %', 'User %']
]

issue_summary

,Issue,Tweets,Users,Tweets per User,Avg Similarity,Tweet %,User %
0,Democracy in the U.S.,91164,82224,1.11,0.550,36.47,35.68
1,Crime,25422,23300,1.09,0.548,10.17,10.11
2,Immigration,18946,16550,1.14,0.563,7.58,7.18
3,Types of Supreme Court justices candidates would pick,17227,16419,1.05,0.541,6.89,7.12
4,The federal budget deficit,13864,13342,1.04,0.534,5.55,5.79
5,Situation in Middle East between Israelis and Palestinians,11432,9634,1.19,0.550,4.57,4.18
6,Gun policy,9911,9304,1.07,0.541,3.96,4.04
7,Relations with China,8768,8459,1.04,0.530,3.51,3.67
8,Climate change,7585,7269,1.04,0.531,3.03,3.15
9,Transgender rights,6497,6233,1.04,0.528,2.60,2.70


In [19]:
try:
    issue_summary.to_csv("../data/issue_summary.csv", index=False)
    print("Saved dataset: ../data/issue_summary.csv")
except Exception as e:
    print("Error while saving:", e)

Saved dataset: ../data/issue_summary.csv


In [20]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

issue_counts = df['issue_semantic'].value_counts().reset_index()
issue_counts.columns = ['Issue', 'Tweet Count']

fig = px.bar(
    issue_counts.sort_values('Tweet Count'),
    x='Tweet Count',
    y='Issue',
    orientation='h',
    text='Tweet Count',
    title='Tweets per Issue (Semantic Classification)'
)

fig.update_layout(
    yaxis=dict(categoryorder='total ascending'),
    height=600
)

fig.show()

In [21]:
top_issues = issue_counts.sort_values('Tweet Count', ascending=False).head(10)

fig = px.bar(
    top_issues.sort_values('Tweet Count'),
    x='Tweet Count',
    y='Issue',
    orientation='h',
    text='Tweet Count',
    title='Top 10 Issues Discussed on Twitter'
)

fig.show()